# Semantic Link - DAX Dependency Graph & Complexity Audit

**Author:** vestergaardj
**Date:** April 2026  
**Runtime:** Microsoft Fabric Notebook (PySpark / Python 3.10+)

---

## Abstract

### The Problem

DAX measures in Power BI semantic models frequently reference other measures, creating
**invisible dependency chains** that are difficult to reason about. As models grow organically
over months and years, developers progressively lose track of:

- **Which measures are actually consumed** by reports versus sitting unused ("dead weight")
- **Where circular references lurk** - these can cause unexpected evaluation behavior and
  performance degradation in the VertiPaq engine
- **Which measures carry the highest complexity burden** - deeply nested `CALCULATE` chains,
  heavy filter-context manipulation, and long transitive dependency paths

There is no built-in Power BI tooling that surfaces this information in an usable way.

### The Solution

This notebook uses **[Semantic Link](https://learn.microsoft.com/fabric/data-science/semantic-link-overview)**
(`sempy.fabric`) to extract every measure from a semantic model, parse the DAX expressions
to detect inter-measure references, and build a **directed dependency graph** using NetworkX.

It then performs five analyzes:

| # | Analysis | Output |
|---|---|---|
| 1 | **Dead measure detection** | List of measures never referenced by any other measure |
| 2 | **Root measure detection** | Foundation measures with zero outgoing dependencies |
| 3 | **Circular reference detection** | All cycles in the dependency graph |
| 4 | **Multi-dimensional complexity scoring** | Weighted score per measure (CALCULATE depth, nesting, branching, filters, dependency depth, fan-out) |
| 5 | **Visual dependency DAG** | Static matplotlib graph + optional interactive pyvis HTML graph |

Everything is brought together in a **ranked audit report** with risk ratings
(Critical / High / Medium / Low) to guide refactoring efforts.

### Why This Tool Wins

- **Creative**: Combines graph theory with heuristic scoring - not just a list, but a
  visual and quantitative audit
- **Reusable**: Works with *any* semantic model in *any* workspace - just change two config values
- **Practical**: Produces a prioritized report that tells you exactly where to focus
- **Developer-friendly**: Runs in a Fabric notebook with zero setup beyond Semantic Link

## Prerequisites

Before running this notebook, ensure the following requirements are met.

| Requirement | Details |
|---|---|
| **Runtime** | Microsoft Fabric Notebook (PySpark / Python). This notebook is designed to run inside Microsoft Fabric - it relies on Fabric's built-in authentication for `sempy.fabric` API calls. |
| **Core libraries** | `semantic-link` (provides `sempy.fabric`), `networkx` (graph data structure and algorithms), `matplotlib` (static graph visualization) |
| **Optional libraries** | `pyvis` - enables an interactive, zoomable HTML dependency graph rendered inline. If not installed, the static matplotlib graph is still produced. |
| **Permissions** | You need **ReadWrite** (or higher) access to the target semantic model. The `list_measures()` API uses the Tabular Object Model (TOM) under the hood, which requires this permission level. |
| **Semantic model** | Any published Power BI semantic model that contains DAX measures. The tool works regardless of model size - from 5 measures to 500+. |
| **XMLA endpoint** | The workspace must have at least **XMLA read-only** enabled (this is the default for Premium / Fabric capacities). |

> **Note:** If running outside Fabric (e.g. local Jupyter), install `semantic-link-sempy`
> via pip. Authentication will need to be handled manually. Local support is limited.

In [ ]:
# Install dependencies (-q = quiet mode).
# Most of these come pre-installed in Fabric, but this makes sure nothing is missing.
#   semantic-link-sempy : Semantic Link Python SDK (provides sempy.fabric)
#   networkx           : Graph data structures and algorithms
#   matplotlib         : Static graph visualization
#   pyvis              : Interactive HTML graph visualization (optional)

%pip install semantic-link-sempy networkx matplotlib pyvis -q

In [ ]:

# --- Imports ---


import re                              # Regular expressions for DAX parsing

import matplotlib.pyplot as plt        # Static graph rendering (Step 7)
import networkx as nx                  # Directed graph construction & analysis
import pandas as pd                    # DataFrame manipulation throughout
import sempy.fabric as fabric          # Semantic Link SDK - the core API
from IPython.display import HTML, display  # Rich notebook output (tables, HTML)

## Configuration

This is the **only cell you need to edit** to point the tool at your semantic model.

| Parameter | Description |
|---|---|
| `WORKSPACE_NAME` | The name of the Fabric workspace containing your semantic model. Set to `None` to use the **current** workspace (i.e. the workspace of the attached lakehouse or the notebook's workspace). |
| `DATASET_NAME` | The **exact name** of the Power BI semantic model (dataset) you want to analyze. This must match the name shown in the Fabric portal. |

> **Tip:** You can find your semantic model name in the Fabric portal under
> *Workspace -> Semantic models*, or by running `fabric.list_datasets()` in a cell.

In [ ]:

# --- User configuration: edit these two values, then Run All ---


WORKSPACE_NAME = None                      # None = use current workspace
DATASET_NAME   = "YourSemanticModelName"   # Exact name of the semantic model

# Examples
# WORKSPACE_NAME = "Sales Analytics Workspace"
# DATASET_NAME   = "Contoso Sales Model"
# ---

## Step 1 - Extract Measures

Use `sempy.fabric.list_measures()` to retrieve **every measure** defined in the semantic model.

### What this returns

The API returns a pandas DataFrame with one row per measure. Key columns include:

| Column | Description |
|---|---|
| `Table Name` | The parent table the measure belongs to |
| `Measure Name` | The display name of the measure |
| `Measure Expression` | The full DAX expression (the formula text) |
| `Measure Hidden` | Whether the measure is hidden from report authors |
| `Measure Description` | The description metadata (if any) |

### How it works under the hood

`list_measures()` connects to the semantic model via the **Tabular Object Model (TOM)**
over the XMLA endpoint. Fabric handles authentication automatically using the notebook's
identity. No tokens or credentials are needed.

In [ ]:
# Step 1: Retrieve all measures from the semantic model
# This single API call fetches every measure definition, including the full DAX
# expression text. It's the foundation for all subsequent analysis.
measures_df = fabric.list_measures(dataset=DATASET_NAME, workspace=WORKSPACE_NAME)

# Print a quick summary so the user knows the data loaded correctly
print(
    f"Found {len(measures_df)} measures "
    f"across {measures_df['Table Name'].nunique()} tables."
)

# Show the first 10 measures - Table Name, Measure Name, and the DAX expression
display(measures_df[["Table Name", "Measure Name", "Measure Expression"]].head(10))

## Step 2 - Parse Measure Dependencies

This is the **core parsing step**. For each measure's DAX expression, we need to find
which *other* measures it references.

### The challenge

DAX expressions use square brackets `[MeasureName]` to reference both **measures** and
**columns**. The difference is:

- **Column reference**: always preceded by a table name -> `Sales[Amount]`, `'Date Table'[Year]`
- **Measure reference**: stands alone -> `[Total Revenue]`, `[Avg Price]`

### Our approach (3-pass parsing)

1. **Strip string literals** - Remove anything inside `"..."` to avoid matching measure
   names that appear inside string constants (e.g. `"Total Revenue"` in an error message)
2. **Strip comments** - Remove `// single-line` and `/* multi-line */` comments
3. **Extract bracket references** - Find all `[Name]` patterns that are **not** preceded
   by a word character (letter, digit, underscore, or apostrophe). This filters out
   table-qualified column references like `Sales[Amount]`.
4. **Cross-reference** - Only keep matches that exist in our known set of measure names.
   This eliminates any remaining column references that happened to not be table-qualified.

### Limitations

- If a measure shares an exact name with a column and is referenced without table
  qualification, this parser will flag it as a measure reference. This is rare in
  well-designed models.
- Measures referenced via `CALCULATE` filter arguments using `Table[Measure]` syntax
  are correctly excluded (treated as column refs). This is intentional - such usage
  is non-standard and typically indicates a column, not a measure.

In [ ]:
# -- Step 2: Parse each DAX expression to find measure-to-measure references ------

def parse_measure_references(
    expression: str, measure_names: set[str]
) -> list[str]:
    """Extract measure references from a DAX expression.

    This function implements a 3-pass parser that:
      1. Strips string literals to avoid false positives
      2. Strips single-line and multi-line comments
      3. Finds [BracketName] patterns not preceded by a table qualifier

    The results are then cross-referenced against the known set of measure
    names to ensure we only return actual measure dependencies.

    Args:
        expression: The raw DAX expression text from the semantic model.
        measure_names: Complete set of all known measure names in the model.
            Used as a lookup to distinguish measures from columns.

    Returns:
        De-duplicated list of measure names that this expression depends on.
        Returns an empty list if the expression is empty or null.
    """
    # Guard: skip empty or null expressions (some measures may have no DAX)
    if not expression or pd.isna(expression):
        return []

    # Pass 1: Remove string literals
    # DAX strings are delimited by double quotes: "some text"
    # We remove them entirely to avoid matching measure names inside strings.
    # Example: FORMAT([Date], "Total Revenue") should NOT match [Total Revenue]
    cleaned = re.sub(r'"[^"]*"', '', expression)

    # Pass 2: Remove comments
    # Single-line comments: everything after // until end of line
    cleaned = re.sub(r'//[^\n]*', '', cleaned)
    # Multi-line comments: everything between /* and */ (DOTALL for newlines)
    cleaned = re.sub(r'/\*.*?\*/', '', cleaned, flags=re.DOTALL)

    # Pass 3: Extract bracket references
    # Pattern: [SomeName] where the opening bracket is NOT preceded by a word
    # character (a-z, 0-9, _, ').  This excludes table-qualified references
    # like Sales[Amount] or 'My Table'[Column].
    #
    # Breakdown of the regex:
    #   (?<![a-zA-Z0-9_'])  - negative lookbehind: no word char before [
    #   \[                  - literal opening bracket
    #   ([^\]]+)            - capture group: one or more chars that are not ]
    #   \]                  - literal closing bracket
    pattern = r"(?<![a-zA-Z0-9_'])\[([^\]]+)\]"
    matches = re.findall(pattern, cleaned)

    # -- Cross-reference against known measures ------------------------------
    # Only keep matches that are actual measure names in this model.
    # This filters out any unqualified column references that slipped through.
    return list({m for m in matches if m in measure_names})


# Build the complete set of all measure names for O(1) lookup
all_measure_names: set[str] = set(measures_df["Measure Name"])

# Apply the parser to every measure's DAX expression.
# The result is a new column "References" containing a list of measure names.
measures_df["References"] = measures_df["Measure Expression"].apply(
    lambda expr: parse_measure_references(expr, all_measure_names)
)

# Preview: show measures sorted by number of outgoing references
# This immediately highlights the most "dependent" measures in the model.
ref_summary = measures_df[["Table Name", "Measure Name", "References"]].copy()
ref_summary["Ref Count"] = ref_summary["References"].apply(len)
display(ref_summary.sort_values("Ref Count", ascending=False).head(15))

## Step 3 - Build the Dependency Graph

Now that we know which measures reference which other measures, we can build a
**directed graph** (DiGraph) using [NetworkX](https://networkx.org/).

### Graph semantics

- **Nodes** = measures. Each node stores metadata: the table name and the DAX expression.
- **Edges** = dependencies. An edge **A -> B** means *"measure A's DAX expression references
  measure B"*. In other words, A *depends on* B.

### Why a directed graph?

Direction matters. If A uses B, that's a very different relationship than B using A.
The direction lets us compute:

- **In-degree** (fan-in): how many other measures depend on *this* measure.
  High fan-in = "hub" measure - breaking it would cascade widely.
- **Out-degree** (fan-out): how many other measures *this* measure depends on.
  High fan-out = complex composition - may be hard to maintain.
- **Paths & chains**: the longest chain of transitive dependencies from any measure.
- **Cycles**: circular references where A -> B -> ... -> A.

In [ ]:
# Step 3: Build a NetworkX directed graph from the parsed dependencies

def build_dependency_graph(measures_df: pd.DataFrame) -> nx.DiGraph:
    """Build a directed graph of measure-to-measure dependencies.

    Creates a NetworkX DiGraph where:
      - Each node represents a measure (keyed by measure name)
      - Each node stores metadata: table name and DAX expression
      - Each directed edge A -> B means "A's expression references B"

    Args:
        measures_df: DataFrame produced by Steps 1-2, containing at minimum:
            'Measure Name', 'Table Name', 'Measure Expression', 'References'.

    Returns:
        A NetworkX DiGraph with all measures as nodes and dependency edges.
    """
    G = nx.DiGraph()

    # Add all measures as nodes
    # We add every measure, even those with no references, so isolated
    # measures still appear in the graph (important for dead-measure detection).
    for _, row in measures_df.iterrows():
        G.add_node(
            row["Measure Name"],
            table=row["Table Name"],
            expression=row["Measure Expression"],
        )

    # Add directed edges for each dependency
    # Edge direction: referencing measure -> referenced measure
    # Example: if "Profit Margin" references "Total Revenue", we add
    #          an edge "Profit Margin" -> "Total Revenue"
    for _, row in measures_df.iterrows():
        for ref in row["References"]:
            if ref in G.nodes:  # Safety check: only add edges to known nodes
                G.add_edge(row["Measure Name"], ref)

    return G


# Build the graph from our parsed measures
graph = build_dependency_graph(measures_df)

# Print summary statistics
print(
    f"Dependency graph: {graph.number_of_nodes()} measures, "
    f"{graph.number_of_edges()} dependency edges"
)
print(f"  Average dependencies per measure: "
      f"{graph.number_of_edges() / max(graph.number_of_nodes(), 1):.1f}")

## Step 4 - Detect Dead & Root Measures

Using the graph's in-degree and out-degree, we can immediately classify measures into
two important categories:

### Dead (unreferenced) measures - `in_degree == 0`

These measures have **zero incoming edges**, meaning no other measure in the model
references them. They fall into two sub-categories:

- **Top-level report measures**: Used directly in report visuals but not by other measures.
  These are expected and healthy.
- **Truly unused measures**: Not referenced by any measure AND not used in any report.
  These are candidates for removal.

> **Caveat:** This analysis only checks measure-to-measure references. A measure with
> zero in-degree might still be used directly in a report visual. Cross-referencing with
> `fabric.list_reports()` can help distinguish the two cases (see Cleanup section).

### Root (base) measures - `out_degree == 0`

These measures have **zero outgoing edges** - their DAX expressions don't reference
any other measure. They sit at the "leaves" of the dependency tree and are typically
simple aggregations like `SUM(Sales[Amount])` or `COUNTROWS(Customers)`.

Root measures form the **foundation** of the dependency graph. If a root measure is
incorrect, every measure that (transitively) depends on it will also be incorrect.

In [ ]:
# -- Step 4: Identify dead (unreferenced) and root (no-dependency) measures -------

def find_dead_measures(G: nx.DiGraph) -> list[str]:
    """Find measures with zero incoming edges (never referenced by others).

    These are either top-level report measures or truly unused "dead" measures.
    In-degree of 0 means NO other measure's DAX expression references this one.
    """
    return [n for n in G.nodes if G.in_degree(n) == 0]


def find_root_measures(G: nx.DiGraph) -> list[str]:
    """Find measures with zero outgoing edges (no dependencies on other measures).

    These are the foundation of the dependency tree - typically simple
    aggregations like SUM(), COUNTROWS(), etc. with no measure references.
    """
    return [n for n in G.nodes if G.out_degree(n) == 0]


# Compute both lists
dead_measures = find_dead_measures(graph)
root_measures = find_root_measures(graph)

# Display results
print(f"Potentially unused measures (never referenced): {len(dead_measures)}")
for m in sorted(dead_measures):
    print(f"   - {m}")

print(f"\nRoot/base measures (no dependencies): {len(root_measures)}")
for m in sorted(root_measures):
    print(f"   - {m}")

## Step 5 - Detect Circular References

Circular dependency chains occur when measure A references B, B references C, and C
references A (or any length of cycle). While the DAX engine can technically evaluate
some circular patterns, they often indicate:

- **Logical errors** in the model design
- **Performance problems** - the engine may need multiple passes to resolve values
- **Maintenance nightmares** - understanding the evaluation order becomes impossible

### How we detect them

NetworkX provides `simple_cycles()` which finds all **elementary cycles** in a directed
graph using Johnson's algorithm. Each cycle is returned as a list of node names forming
the loop.

> **Healthy result:** "No circular dependencies detected." - This means the dependency
> graph is a true DAG (Directed Acyclic Graph), which is the ideal state.

In [ ]:
# Step 5: Detect circular dependency chains
# Uses Johnson's algorithm via NetworkX to find all elementary cycles.
# A cycle is a path like A -> B -> C -> A where you end up back at the start.

def detect_circular_references(G: nx.DiGraph) -> list[list[str]]:
    """Return all circular dependency chains in the measure graph.

    Uses nx.simple_cycles() - Johnson's algorithm for finding all
    elementary circuits in a directed graph. Returns a list of cycles,
    where each cycle is a list of measure names forming the loop.
    """
    return list(nx.simple_cycles(G))


# Find all cycles in the dependency graph
cycles = detect_circular_references(graph)

# Display results
if cycles:
    # WARNING: Circular references found - these should be investigated
    print(f"WARNING: Found {len(cycles)} circular dependency chain(s):")
    for i, cycle in enumerate(cycles, 1):
        # Show the full chain as A -> B -> C -> A (repeating the start node)
        chain = " -> ".join(cycle)
        print(f"   Cycle {i}: {chain} -> {cycle[0]}")
else:
    # Clean graph - no cycles. This is the ideal state.
    print("No circular dependencies detected.")

## Step 6 - Score Measure Complexity

Not all measures are equally complex. A simple `SUM()` is trivial; a deeply nested
`CALCULATE` with multiple filter manipulations inside a `SWITCH` is a maintenance hazard.

This step assigns a **multi-dimensional complexity score** to each measure using both
**DAX expression analysis** and **graph-based metrics**.

### Scoring dimensions

| Metric | Weight | What it captures | Why it matters |
|---|---|---|---|
| `CALCULATE` / `CALCULATETABLE` count | **3** | Context-transition calls | Each CALCULATE changes the filter context - the trickiest and most error-prone DAX pattern |
| Max parenthesis nesting depth | **1** | Expression readability | Deeply nested expressions are hard to read, debug, and maintain |
| Branching (`IF` / `SWITCH`) | **2** | Conditional logic | Branching increases the number of code paths to test |
| Filter functions (`FILTER`, `ALL`, `ALLEXCEPT`, `ALLSELECTED`, `REMOVEFILTERS`) | **2** | Filter-context manipulation | These functions modify evaluation context - misuse causes subtle bugs |
| Dependency depth (longest path in graph) | **2** | Transitive dependency chain | A measure at the end of a long chain amplifies errors from upstream |
| Fan-out (direct dependencies) | **1** | Composition breadth | Measures referencing many others are harder to reason about |

### Composite score

The **total complexity score** is a weighted sum of all dimensions. Higher = more complex
= harder to maintain. The weights were chosen to emphasise context-transition complexity
(`CALCULATE` carries the most risk in DAX) while still rewarding simplicity across all
other dimensions.

### Dependency depth calculation

The *dependency depth* of a measure is the **longest path** from that measure through the
dependency graph. This is computed via a memoised depth-first search (DFS).  For example:

```
MeasureA -> MeasureB -> MeasureC -> MeasureD
```

Here, MeasureA has a dependency depth of 3 (the longest chain from A to a root).

In [ ]:
# Step 6: Score each measure's complexity across multiple dimensions

def _longest_path_from(G: nx.DiGraph, source: str) -> int:
    """Compute the longest path length from a source node via memoised DFS.

    Walks the graph's successors (outgoing edges) recursively, tracking the
    maximum depth encountered. Uses memoisation to avoid redundant traversals.

    The `memo[node] = 0` guard before recursion handles cycles correctly -
    if we encounter a node already being processed (in-stack), we treat its
    depth as 0 to break the infinite loop.

    Args:
        G: The dependency DiGraph.
        source: The node to start the DFS from.

    Returns:
        Length of the longest path from source to any reachable leaf node.
        Returns 0 if the source has no outgoing edges (it is a root measure).
    """
    memo: dict[str, int] = {}

    def dfs(node: str) -> int:
        if node in memo:
            return memo[node]
        # Set to 0 BEFORE recursing - acts as a cycle guard.
        # If we encounter this node again during DFS, we return 0
        # instead of recursing infinitely.
        memo[node] = 0
        max_len = 0
        for succ in G.successors(node):
            max_len = max(max_len, 1 + dfs(succ))
        memo[node] = max_len
        return max_len

    return dfs(source)


def score_complexity(
    expression: str, name: str, G: nx.DiGraph
) -> dict:
    """Score the complexity of a single DAX measure across multiple dimensions.

    Analyzes both the DAX expression text (via regex pattern matching) and
    the measure's position in the dependency graph (via graph metrics).

    Args:
        expression: The raw DAX expression text.
        name: The measure name (used for graph lookups).
        G: The dependency DiGraph built in Step 3.

    Returns:
        Dictionary containing individual metric values and a weighted
        composite 'Complexity Score'. Keys:
          - 'CALCULATE Count', 'Max Nesting Depth', 'Branching (IF/SWITCH)',
            'Filter Functions', 'Dependency Depth', 'Fan-Out', 'Fan-In',
            'Complexity Score'
    """
    # Handle empty/null expressions (some measures may not have DAX)
    if not expression or pd.isna(expression):
        expression = ""

    # Work with uppercase for case-insensitive DAX function matching
    expr_upper = expression.upper()

    # Metric 1: CALCULATE / CALCULATETABLE count (weight: 3)
    # These are commonly misused DAX functions.
    # Each call transitions the filter context, which is the #1 source of
    # DAX bugs in complex models.
    calculate_count = len(
        re.findall(r"\bCALCULATE(?:TABLE)?\s*\(", expr_upper)
    )

    # Metric 2: Maximum parenthesis nesting depth (weight: 1)
    # Walk through the expression tracking paren depth.
    # Deeply nested expressions are objectively harder to read and debug.
    depth = max_depth = 0
    for ch in expression:
        if ch == "(":
            depth += 1
            max_depth = max(max_depth, depth)
        elif ch == ")":
            depth -= 1

    # -- Metric 3: Branching - IF / SWITCH calls (weight: 2) ----------------
    # Each branch increases the number of code paths, making the measure
    # harder to test and reason about.
    branching = len(re.findall(r"\b(?:IF|SWITCH)\s*\(", expr_upper))

    # -- Metric 4: Filter-context manipulation functions (weight: 2) ---------
    # These functions modify the evaluation context - misuse leads to subtle
    # calculation bugs that are extremely hard to diagnose.
    filter_funcs = len(
        re.findall(
            r"\b(?:FILTER|ALL|ALLEXCEPT|ALLSELECTED|"
            r"ALLNOBLANKROW|REMOVEFILTERS)\s*\(",
            expr_upper,
        )
    )

    # -- Metric 5 & 6: Graph-based metrics -----------------------------------
    # Fan-out (weight: 1) = number of measures this expression directly uses
    fan_out = G.out_degree(name) if name in G else 0
    # Fan-in (informational) = number of measures that depend on this one
    fan_in = G.in_degree(name) if name in G else 0

    # Metric 7: Dependency depth (weight: 2)
    # Longest chain of transitive dependencies from this measure.
    # A measure at the end of a 5-deep chain amplifies upstream errors.
    dep_depth = _longest_path_from(G, name) if name in G else 0

    # Compute weighted composite score
    # Weights reflect relative risk: CALCULATE (3) > branching/filters/depth (2) > nesting/fan-out (1)
    total = (
        calculate_count * 3   # Context-transition risk
        + max_depth * 1       # Readability penalty
        + branching * 2       # Conditional complexity
        + filter_funcs * 2    # Filter-context manipulation
        + dep_depth * 2       # Transitive dependency chain
        + fan_out * 1         # Composition breadth
    )

    return {
        "CALCULATE Count": calculate_count,
        "Max Nesting Depth": max_depth,
        "Branching (IF/SWITCH)": branching,
        "Filter Functions": filter_funcs,
        "Dependency Depth": dep_depth,
        "Fan-Out": fan_out,
        "Fan-In": fan_in,
        "Complexity Score": total,
    }


# Apply scoring to every measure
complexity_records = []
for _, row in measures_df.iterrows():
    scores = score_complexity(row["Measure Expression"], row["Measure Name"], graph)
    scores["Measure"] = row["Measure Name"]
    scores["Table"] = row["Table Name"]
    complexity_records.append(scores)

# Build a DataFrame sorted by complexity score (most complex first)
complexity_df = pd.DataFrame(complexity_records).sort_values(
    "Complexity Score", ascending=False
)

# Display the top 20 most complex measures
display(
    complexity_df[
        [
            "Measure", "Table", "Complexity Score", "CALCULATE Count",
            "Max Nesting Depth", "Branching (IF/SWITCH)",
            "Filter Functions", "Dependency Depth", "Fan-Out", "Fan-In",
        ]
    ].head(20)
)

## Step 7 - Visualize the Dependency DAG

A picture is worth a thousand rows. This step renders the dependency graph as a
**color-coded, size-coded static image** using matplotlib.

### Visual encoding

| Visual property | Maps to | Interpretation |
|---|---|---|
| **Node color** | Complexity score | Green = low complexity, Yellow = medium, Red = high complexity. Uses the `RdYlGn_r` (reversed Red-Yellow-Green) colormap. |
| **Node size** | Fan-in (in-degree) | Larger nodes are referenced by more other measures - they are "hub" measures whose correctness is critical. |
| **Edge direction** | Dependency | Arrow from A to B means A's DAX references measure B. |
| **Labels** | Measure name | Shown when the graph has ≤ 60 nodes (configurable). For larger models, labels are hidden to avoid clutter. |

### Layout algorithm

The code first attempts a **hierarchical (dot) layout** via Graphviz, which produces
the clearest DAG visualization with dependency levels flowing top-to-bottom. If Graphviz
is not installed, it falls back to a **spring (force-directed) layout** from NetworkX.

In [ ]:
# -- Step 7: Render a static color-coded dependency graph -----------------------

def visualize_dependency_graph(
    G: nx.DiGraph,
    complexity_df: pd.DataFrame,
    max_labels: int = 60,
) -> None:
    """Render the dependency graph as a color-coded DAG using matplotlib.

    Node color encodes complexity score (green->yellow->red).
    Node size encodes fan-in (how many other measures depend on it).
    Labels are shown only for graphs with ≤ max_labels nodes.

    Args:
        G: The dependency DiGraph built in Step 3.
        complexity_df: Complexity scoring DataFrame from Step 6.
        max_labels: Maximum number of nodes before labels are hidden.
            Set to 0 to always hide labels, or a large number to always show.
    """
    if G.number_of_nodes() == 0:
        print("No measures to visualize.")
        return

    # Map each measure to its complexity score
    score_map = dict(
        zip(complexity_df["Measure"], complexity_df["Complexity Score"])
    )
    max_score = max(score_map.values(), default=1) or 1  # Avoid division by zero

    # Color: RdYlGn_r colormap (green=low, red=high complexity)
    cmap = plt.cm.RdYlGn_r
    node_colors = [
        cmap(score_map.get(n, 0) / max_score) for n in G.nodes
    ]

    # -- Size: base size + fan-in bonus (hub measures are larger) ------------
    node_sizes = [300 + G.in_degree(n) * 150 for n in G.nodes]

    # Layout: prefer hierarchical (dot) layout, fall back to spring
    try:
        # Graphviz "dot" produces a top-to-bottom hierarchy - ideal for DAGs
        pos = nx.nx_agraph.graphviz_layout(G, prog="dot")
    except Exception:
        # Graphviz not installed - use spring (force-directed) layout instead
        pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

    # Draw the graph
    fig, ax = plt.subplots(figsize=(16, 10))

    # Draw edges first (so they appear behind nodes)
    nx.draw_networkx_edges(
        G, pos, ax=ax, edge_color="#cccccc",
        arrows=True, arrowsize=12, alpha=0.6,
    )
    # Draw nodes on top
    nx.draw_networkx_nodes(
        G, pos, ax=ax, node_color=node_colors,
        node_size=node_sizes, edgecolors="#666666", linewidths=0.5,
    )

    # Draw labels only if the graph isn't too crowded
    if G.number_of_nodes() <= max_labels:
        nx.draw_networkx_labels(
            G, pos, ax=ax, font_size=7, font_weight="bold",
        )

    # Add color bar legend
    sm = plt.cm.ScalarMappable(
        cmap=cmap, norm=plt.Normalize(0, max_score)
    )
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.6)
    cbar.set_label("Complexity Score", fontsize=10)

    # Title and formatting
    ax.set_title(
        "DAX Measure Dependency Graph\n"
        "(color = complexity, size = fan-in)",
        fontsize=14, fontweight="bold",
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()


# Render the graph
visualize_dependency_graph(graph, complexity_df)

### Optional - Interactive HTML Graph (pyvis)

The static matplotlib graph above is great for an overview, but for **large models**
(50+ measures) you often need to zoom, pan, and hover over nodes to explore the graph.

[**pyvis**](https://pyvis.readthedocs.io/) renders an interactive, physics-based graph
directly in the notebook output using JavaScript (vis.js under the hood).

### Interactive features

- **Drag** nodes to rearrange the layout
- **Scroll** to zoom in/out
- **Hover** over a node to see its name, table, and complexity score
- **Click and drag** the canvas to pan
- Node colors and sizes use the same encoding as the static graph above

> **Note:** The graph is rendered inline using an iframe with embedded HTML.
> No external file is created - everything stays within the notebook output.

In [ ]:
# Optional: Interactive pyvis graph rendered inline

def visualize_interactive(
    G: nx.DiGraph,
    complexity_df: pd.DataFrame,
) -> None:
    """Render an interactive, zoomable dependency graph inline using pyvis.

    The graph is rendered as embedded HTML within an iframe - no external file
    is created. Nodes are color-coded (green->red for complexity) and sized
    by fan-in. Hovering shows measure metadata.

    Args:
        G: The dependency DiGraph built in Step 3.
        complexity_df: Complexity scoring DataFrame from Step 6.
    """
    # Import pyvis (handle missing install)
    try:
        from pyvis.network import Network
    except ImportError:
        print("pyvis is not installed. Run: %pip install pyvis")
        return

    # Build score lookup
    score_map = dict(
        zip(complexity_df["Measure"], complexity_df["Complexity Score"])
    )
    max_score = max(score_map.values(), default=1) or 1

    # Configure the pyvis network
    net = Network(
        height="700px",           # Canvas height in the iframe
        width="100%",             # Full width
        directed=True,            # Show arrow heads on edges
        notebook=True,            # Enable notebook rendering mode
        cdn_resources="in_line",  # Embed all JS/CSS inline (no external CDN)
    )
    # Use Barnes-Hut physics for a nice spread-out layout
    net.barnes_hut(gravity=-3000, spring_length=150)

    # Helper: map a normalised 0-1 score to a green->red hex color
    def score_to_hex(score: float) -> str:
        """Convert a 0-1 normalised score to a hex color string.

        0.0 = pure green (#00ff00), 0.5 = yellow (#ffff00), 1.0 = pure red (#ff0000).
        Uses linear interpolation in the red and green channels.
        """
        r = int(255 * min(score * 2, 1))
        g = int(255 * min((1 - score) * 2, 1))
        return f"#{r:02x}{g:02x}00"

    # Add nodes with color, size, and hover tooltip
    for node in G.nodes:
        norm = score_map.get(node, 0) / max_score
        table = G.nodes[node].get("table", "")
        net.add_node(
            node,
            label=node,                               # Visible label
            title=(                                    # Hover tooltip
                f"{node}\nTable: {table}\n"
                f"Score: {score_map.get(node, 0)}"
            ),
            color=score_to_hex(norm),                  # Green->Red
            size=10 + G.in_degree(node) * 5,           # Bigger = more dependents
        )

    # Add directed edges
    for src, tgt in G.edges:
        net.add_edge(src, tgt)

    # Render inline via iframe with srcdoc
    # We generate the full HTML string, then embed it in an iframe's srcdoc
    # attribute. This avoids writing a file to disk (which doesn't work
    # reliably in Fabric) and renders directly in the notebook output.
    html = net.generate_html()
    display(HTML(
        f'<iframe srcdoc="{html.replace(chr(34), "&quot;")}" '
        f'width="100%" height="750px" frameborder="0"></iframe>'
    ))


# Render the interactive graph
visualize_interactive(graph, complexity_df)

## Step 8 - Audit Summary Report

This final step consolidates **every analysis performed above** into a single, prioritized
audit table - the main deliverable of this notebook.

### What the report contains

| Column | Source | Meaning |
|---|---|---|
| **Measure / Table** | Step 1 | The measure name and its parent table |
| **Complexity Score** | Step 6 | Weighted composite score (higher = riskier) |
| **CALCULATE Count** | Step 6 | Raw count of `CALCULATE` calls in the expression |
| **Dependency Depth** | Step 6 | Longest downstream dependency chain |
| **Fan-Out / Fan-In** | Step 3 | Number of measures this measure references / is referenced by |
| **Unreferenced** | Step 4 | `True` if the measure has zero incoming edges ("dead" measure) |
| **Circular Ref** | Step 5 | `True` if the measure participates in any cycle |
| **Risk** | Computed | Risk rating derived from the score + circular flag |

### Risk-rating logic

```
+--------------------------------------------------+
|  Circular reference?  --> YES --> "Critical"     |
|        | NO                                       |
|  Complexity ≥ 20?     --> YES --> "High"         |
|        | NO                                       |
|  Complexity ≥ 10?     --> YES --> "Medium"       |
|        | NO                                       |
|  Otherwise            ----------> "Low"           |
+--------------------------------------------------+
```

The report is sorted **risk-first** (Critical -> High -> Medium -> Low), then by
descending complexity score within each tier. A summary banner is printed above
the table for a quick health snapshot.

In [ ]:
# Step 8: Audit Summary Report
#
# This cell defines `generate_audit_report()` which merges all prior analyzes
# (complexity scores, dead/root flags, cycle membership) into a single
# prioritized DataFrame.  It also prints a summary banner for quick scanning.
# ---


def generate_audit_report(
    complexity_df: pd.DataFrame,
    dead_measures: list[str],
    root_measures: list[str],
    cycles: list[list[str]],
    graph: nx.DiGraph,
) -> pd.DataFrame:
    """Build a full audit report for every measure.

    The report adds boolean flags for "unreferenced" and "circular reference",
    computes a risk rating (Critical / High / Medium / Low), and sorts the
    table so the most urgent issues appear first.

    Args:
        complexity_df: Complexity scoring DataFrame from Step 6.
        dead_measures: Measure names with zero incoming edges (Step 4).
        root_measures: Measure names with zero outgoing edges (Step 4).
        cycles: Circular dependency chains detected in Step 5.
        graph: The dependency DiGraph from Step 3.

    Returns:
        A DataFrame sorted by risk (Critical -> Low) then by descending
        complexity score, ready for display or export.
    """

    # Start with a copy of the complexity DataFrame to avoid mutating the
    # original - keeps everything reproducible.
    report = complexity_df.copy()

    # Boolean flag columns
    # "Unreferenced" = True if the measure is never consumed by another
    # measure (dead code candidate).
    report["Unreferenced"] = report["Measure"].isin(dead_measures)

    # "Is Root" = True if the measure has no outgoing dependency edges,
    # meaning it is a leaf-level base measure that everything else builds on.
    report["Is Root"] = report["Measure"].isin(root_measures)

    # "Circular Ref" = True if the measure appears in *any* cycle.
    # Flatten the list-of-lists into a single set of member names.
    cycle_members = {m for cycle in cycles for m in cycle}
    report["Circular Ref"] = report["Measure"].isin(cycle_members)

    # Risk classification function
    def _risk(row):
        """Assign a risk rating based on cycle membership and score.

        Priority order:
            1. Circular reference -> "Critical"  (always highest urgency)
            2. Score ≥ 20        -> "High"
            3. Score ≥ 10        -> "Medium"
            4. Everything else   -> "Low"
        """
        if row["Circular Ref"]:
            return "Critical"
        if row["Complexity Score"] >= 20:
            return "High"
        if row["Complexity Score"] >= 10:
            return "Medium"
        return "Low"

    # Apply the risk function row-by-row and store in a new column.
    report["Risk"] = report.apply(_risk, axis=1)

    # Sort: most severe first, then by score descending
    # We map risk labels to integers for deterministic sort ordering.
    risk_order = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3}
    report["_risk_sort"] = report["Risk"].map(risk_order)
    report = report.sort_values(
        ["_risk_sort", "Complexity Score"], ascending=[True, False]
    ).drop(columns=["_risk_sort"])      # Drop the helper column after sorting

    return report


# Generate the report
report_df = generate_audit_report(
    complexity_df, dead_measures, root_measures, cycles, graph
)

# Summary banner
# A quick-glance health check printed above the table.
print("=" * 60)
print("  DAX DEPENDENCY GRAPH - AUDIT SUMMARY")
print("=" * 60)
print(f"  Total measures:            {len(report_df)}")
print(f"  Dependency edges:          {graph.number_of_edges()}")
print(f"  Unreferenced measures:     {report_df['Unreferenced'].sum()}")
print(f"  Root measures (no deps):   {report_df['Is Root'].sum()}")
print(f"  Circular references:       {len(cycles)}")
print(f"  High complexity (>=20):    {(report_df['Complexity Score'] >= 20).sum()}")
print(f"  Medium complexity (10-19): {report_df['Complexity Score'].between(10, 19).sum()}")
print("=" * 60)

# Display the top 30 most critical rows
# Only the most relevant columns are shown.  The full DataFrame
# (report_df) is still available for further analysis or export.
display(
    report_df[
        [
            "Measure", "Table", "Risk", "Complexity Score",
            "CALCULATE Count", "Dependency Depth", "Fan-Out", "Fan-In",
            "Unreferenced", "Circular Ref",
        ]
    ].head(30)
)

## Cleanup & Next Steps

You've completed the full audit. Below is guidance for interpreting results and
acting on findings.

---

### Interpreting the Risk Ratings

| Risk | Threshold | Recommended Action |
|---|---|---|
| **Critical** | Measure participates in a **circular reference** | Investigate *immediately* - break the dependency cycle. Circular refs cause unpredictable evaluation order and potential runtime errors. |
| **High** | Complexity Score **≥ 20** (no cycle) | Schedule a refactoring session. Split deeply nested `CALCULATE` chains into smaller, well-named intermediate measures. |
| **Medium** | Complexity Score **10 - 19** | Review during the next planned refactoring pass. These measures are maintainable today but may drift toward "High" over time. |
| **Low** | Complexity Score **< 10**, no cycle | No action needed. These measures are simple, well-scoped, and easy to maintain. |

### Reading the Dependency Graph

* **Hub measures** (high fan-in) are referenced by many others - changes ripple
  widely. Treat them as critical infrastructure and document them thoroughly.
* **Leaf measures** (high fan-out, zero fan-in) sit at the top of the DAG and
  tend to be the most complex. They are user-facing report measures.
* **Isolated nodes** (zero fan-in *and* zero fan-out) reference no other
  measures and are not referenced - strong candidates for cleanup.

### Suggested Follow-Ups

1. **Prune dead measures** - Unreferenced measures with no report usage can
   likely be removed. Cross-reference with `sempy.fabric.list_reports()` and
   check report visuals before deleting.
2. **Simplify high-complexity measures** - Extract reusable filter logic into
   intermediate measures. A good rule of thumb: if a DAX expression exceeds
   15 lines or contains more than 2 nested `CALCULATE` calls, it benefits
   from decomposition.
3. **Document hub measures** - Use the fan-in count to identify "hub" measures
   that many others depend on. Add `Measure Description` metadata in Tabular
   Editor or via the `TOM` object model.
4. **Schedule regular audits** - Re-run this notebook periodically (e.g. after
   each sprint or release) to track complexity trends over time. Compare
   successive `report_df` outputs to spot regression.
5. **Export for stakeholders** - Convert `report_df` to Excel or CSV for
   sharing with BI leads:
   ```python
   report_df.to_csv("/lakehouse/default/Files/dax_audit_report.csv", index=False)
   ```

---

### Limitations & Caveats

* **Column references are excluded by design** - The parser only resolves
  *measure-to-measure* dependencies. Column-level lineage (e.g.
  `Sales[Amount]`) is left out on purpose to keep the graph focused.
* **Dynamic measure names** - Measures referenced via `SELECTEDMEASURE()` or
  `SWITCH` over a disconnected table cannot be resolved statically.
* **Calculation groups** - Calculation items are not modeled as nodes. If your
  model uses calculation groups heavily, some hidden dependencies may not
  appear in the graph.
* **Cross-model references** - Direct Query / composite model references to
  external datasets are out of scope for this tool.

---

*Built with [Semantic Link](https://learn.microsoft.com/fabric/data-science/semantic-link-overview)
for the Fabric Semantic Link Developer Experience Challenge.*